# 07. Packaging a Repeatable Processing Script
**Engineering Data Processing with Python** | one-day course | about 30 minutes

> **How this notebook works**
> * Run every grey code cell in order: click it, then press **Shift + Enter**.
> * Most cells just need running and reading. Cells marked **Your turn** have a blank, `____`, for you to replace. The example just above shows you how.
> * Stuck? Click **Stuck? Click to show the answer** under the cell. Using it is fine.
> * Cells that show only a title are helper code written for you. Just run them.


In [ ]:
# @title Setup: run this cell first (it loads the course data)
# ===================== SETUP: run this cell first (click it, then Shift+Enter) =====================
# It makes sure the course data is available in this Colab session and loads the tools we need.
import os, zipfile, urllib.request
from pathlib import Path

DATA_ZIP_URL = "https://raw.githubusercontent.com/buildyourai/ei-python-data-processing/main/data.zip"   # course data (fictional / public)

def get_course_data():
    if Path("data").is_dir():
        print("Data folder found:", sorted(p.name for p in Path("data").iterdir())[:20])
        return
    if not Path("data.zip").exists() and DATA_ZIP_URL:
        print("Downloading course data...")
        urllib.request.urlretrieve(DATA_ZIP_URL, "data.zip")
    zip_path = Path("data.zip")
    if not zip_path.exists():
        try:
            from google.colab import files
            print("Choose data.zip from your computer (your trainer will tell you where it is).")
            uploaded = files.upload()
            zip_path = Path(next(iter(uploaded)))
        except ImportError:
            raise FileNotFoundError("Put data.zip or the data folder next to this notebook.")
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(".")
    print("Data ready:", sorted(p.name for p in Path("data").iterdir()))

get_course_data()
Path("outputs").mkdir(exist_ok=True)

import pandas as pd
import numpy as np
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

def check(condition, ok="Looks right.", hint="Not quite yet. Re-read the instructions above the cell."):
    """Tiny self-check helper used throughout the course."""
    print(("PASS: " + ok) if condition else ("NOT YET: " + hint))

## The point of this module

A notebook is a good place to work things out. It is a poor thing to hand a colleague with "run this every month":
cells can be run out of order and settings are scattered through the code.

A **script** fixes that. It has three features:

1. **Each step is a function**: load, clean, dedupe, join, check, save. Everything you did in Modules 03 to 06.
2. **Settings live in a separate config file**: file path, column names, spelling lookups, thresholds. A new export
   with different column names means editing the config, not the code.
3. **One command runs everything**, the same way every time, and writes a log.

You will not write the script today. It is provided, built from the steps you already did. You will read its
outline, run it, and change its behaviour **by editing the config only**. That is how you would use it at work.

```
project/
├── pipeline.py          the code (rarely changes)
├── config_cmms.json     settings for one data source
├── data/                raw inputs, never edited
└── outputs/<run_name>/  one folder per run: clean data, issues, report, log
```

In [ ]:
import shutil, json, sys, importlib, inspect
shutil.copy("data/checkpoints/pipeline.py", "pipeline.py")
import pipeline
importlib.reload(pipeline)

print("pipeline.py is ready")

The steps inside `pipeline.py`, in the order they run:

| Function | What it does | You did this in |
|---|---|---|
| `load_work_orders` | Reads the file and renames columns using the config | Module 02 |
| `clean_work_orders` | Tidies IDs, maps types, converts numbers and dates, maps technician names | Module 03 |
| `dedupe_work_orders` | Removes exact copies and keeps the newest version of each work order | Module 04 |
| `join_assets` | Adds asset type and site, labels unknown assets | Module 05 |
| `validate` | Applies the quality rules, splits errors from warnings | Module 06 |
| `save_outputs` | Writes the clean data, KPI-ready data, issues list and report | Module 06 |
| `run_pipeline` | Runs all of the above in order and writes a log | |

Look at one step in full. This is Module 04's "keep the newest version" rule, now inside a function:

In [ ]:
print(inspect.getsource(pipeline.dedupe_work_orders))

## 1. The settings (config)

A Python dictionary saved as a JSON file, the same kind of file as the boiler data dictionary in the primer.

### Your turn 7.1
Fill in the input file (`work_orders_cmms.csv`) and the hours threshold (**80**).

In [ ]:
CONFIG = {
    "run_name": "cmms_full_history",
    "source": {
        "path": "data/____",
        "sep": ",",
        "decimal": ".",
        "date_formats": ["%d/%m/%Y %H:%M", "%Y-%m-%d %H:%M"],
        "modified_formats": ["%Y-%m-%d %H:%M:%S"],
        "column_map": {},
    },
    "na_values": ["", "N/A", "-", "TBC"],
    "type_map": {"PM": "PM", "PREVENTIVE": "PM", "CM": "CM", "CORRECTIVE": "CM", "BREAKDOWN": "CM",
                 "INSP": "INSP", "INSPECTION": "INSP"},
    "staff": ["Aoife Byrne", "Ciarán Walsh", "Declan Murphy", "Niamh O'Connor",
              "Seán Kelly", "Orla Doyle", "Patrick Ryan", "Gráinne Fitzgerald"],
    "technician_map_file": "data/checkpoints/m04_technician_map.csv",
    "asset_register_file": "data/checkpoints/m04_asset_register_dedup.csv",
    "decommissioned_file": "data/asset_register.xlsx",
    "thresholds": {"max_hours": ____, "max_cost": 50000, "open_days": 60, "as_of": "2026-09-30"},
    "error_rules": ["closed_before_raised", "negative_hours", "implausible_hours",
                    "closed_without_date", "unknown_asset", "implausible_cost"],
    "output_dir": "outputs",
}
with open("config_cmms.json", "w", encoding="utf-8") as f:
    json.dump(CONFIG, f, indent=2, ensure_ascii=False)
print("Saved config_cmms.json")

<details><summary><b>Stuck? Click to show the answer</b></summary>

```python
CONFIG = {
    "run_name": "cmms_full_history",
    "source": {
        "path": "data/work_orders_cmms.csv",
        "sep": ",",
        "decimal": ".",
        "date_formats": ["%d/%m/%Y %H:%M", "%Y-%m-%d %H:%M"],
        "modified_formats": ["%Y-%m-%d %H:%M:%S"],
        "column_map": {},
    },
    "na_values": ["", "N/A", "-", "TBC"],
    "type_map": {"PM": "PM", "PREVENTIVE": "PM", "CM": "CM", "CORRECTIVE": "CM", "BREAKDOWN": "CM",
                 "INSP": "INSP", "INSPECTION": "INSP"},
    "staff": ["Aoife Byrne", "Ciarán Walsh", "Declan Murphy", "Niamh O'Connor",
              "Seán Kelly", "Orla Doyle", "Patrick Ryan", "Gráinne Fitzgerald"],
    "technician_map_file": "data/checkpoints/m04_technician_map.csv",
    "asset_register_file": "data/checkpoints/m04_asset_register_dedup.csv",
    "decommissioned_file": "data/asset_register.xlsx",
    "thresholds": {"max_hours": 80, "max_cost": 50000, "open_days": 60, "as_of": "2026-09-30"},
    "error_rules": ["closed_before_raised", "negative_hours", "implausible_hours",
                    "closed_without_date", "unknown_asset", "implausible_cost"],
    "output_dir": "outputs",
}
with open("config_cmms.json", "w", encoding="utf-8") as f:
    json.dump(CONFIG, f, indent=2, ensure_ascii=False)
print("Saved config_cmms.json")
```
</details>

## 2. Run it with one command

`!` runs a command as if typed into a terminal. The line below is the same as typing
`python pipeline.py config_cmms.json` on your own computer.

In [ ]:
!{sys.executable} pipeline.py config_cmms.json

Read the log above: rows loaded, duplicates removed, rules that fired. Now look at what it saved:

In [ ]:
from pathlib import Path
run_dir = Path("outputs/cmms_full_history")
print(sorted(p.name for p in run_dir.iterdir()))
pd.read_csv(run_dir / "dq_report.csv")

In [ ]:
check((run_dir / "dq_report.csv").exists(), "The whole workflow ran from one command.",
      "No outputs. Read the error printed by the run cell.")

## 3. Change a setting, not the code

The maintenance manager wants any job longer than one **12-hour** shift flagged, not 80 hours.

### Your turn 7.2
Give the new run its own name (`cmms_strict`) and set the threshold to **12**. The rest is copied from the first config.

In [ ]:
strict = json.loads(json.dumps(CONFIG))          # a full copy of the settings
strict["run_name"] = "____"
strict["thresholds"]["max_hours"] = ____
with open("config_strict.json", "w", encoding="utf-8") as f:
    json.dump(strict, f, indent=2, ensure_ascii=False)

!{sys.executable} pipeline.py config_strict.json

<details><summary><b>Stuck? Click to show the answer</b></summary>

```python
strict = json.loads(json.dumps(CONFIG))          # a full copy of the settings
strict["run_name"] = "cmms_strict"
strict["thresholds"]["max_hours"] = 12
with open("config_strict.json", "w", encoding="utf-8") as f:
    json.dump(strict, f, indent=2, ensure_ascii=False)

!{sys.executable} pipeline.py config_strict.json
```
</details>

In [ ]:
base = pd.read_csv("outputs/cmms_full_history/dq_report.csv").set_index("rule")["rows_flagged"]
new = pd.read_csv("outputs/cmms_strict/dq_report.csv").set_index("rule")["rows_flagged"]
pd.DataFrame({"80 hours": base, "12 hours": new})

More jobs flagged, and `pipeline.py` was not touched. Whether 12 hours is right is an engineering conversation;
the script lets you have it with numbers, in seconds.

## 4. Testing a piece on its own

Because each step is a function, you can try it on tiny inputs where you know the right answer. Example: the
number converter from Module 03.

In [ ]:
pipeline.to_number(pd.Series(["€1,250.00", "3", "N/A"]))

### Your turn 7.3
Some European systems write one thousand two hundred and fifty euro as `1.250,00` (dot for thousands, comma for
decimals). Tell the converter the decimal mark is a comma: `","`.

In [ ]:
pipeline.to_number(pd.Series(["1.250,00", "2,5"]), decimal="____")

<details><summary><b>Stuck? Click to show the answer</b></summary>

```python
pipeline.to_number(pd.Series(["1.250,00", "2,5"]), decimal=",")
```
</details>

You get 1250.0 and 2.5. You will need this in the capstone.

## 5. Using it outside Colab

On any computer with Python:

```
pip install pandas openpyxl
python pipeline.py config_cmms.json
```

Schedule that line (Windows Task Scheduler, or your IT team's tools) and the monthly clean-up runs itself.

**Optional, with Gemini:** ask it to explain `pipeline.py` function by function, or ask *"How would I add a step to
this script that saves a summary to Excel?"* Before using any code it suggests: does it change anything you didn't
ask for? Does it type in file paths or thresholds that should be in the config?

## Recap
* Script = functions for each step + a config file for everything that varies + one command.
* Change behaviour by editing the config, not the code.
* Test functions on small inputs where you know the answer.

Next: **08 Mini-Capstone**.